# 🎯 ToMBench QLoRA + Cross-Benchmark Transfer Evaluation

## 📐 방법론

### 데이터 분할 (3-way)
```
ToMBench 2860문항
├── Test  858 (30%)  ← 봉인, 최종 평가만
└── 2002 (70%)
    ├── Train 1602 (80%) → QLoRA 학습
    └── Val    400 (20%) → epoch별 성능 체크 / Early Stopping
```

### 실험 흐름
```
                  Qwen2.5-3B (4bit)
                        │
    ┌───────────────────┴────────────────────┐
    │ [1] Baseline 측정 (학습 전, zero-shot) │
    │  → ToMBench/OpenToM/ToMi/SocialIQa/HiToM│
    └───────────────────┬────────────────────┘
                        │
          QLoRA 학습 (Train + Val 모니터링)
                        │
    ┌───────────────────┴────────────────────┐
    │ [2] Finetuned 측정 (학습 후)            │
    │  → 동일한 5개 평가셋                    │
    └───────────────────┬────────────────────┘
                        │
              [3] Δ (개선폭) 비교 표 산출
```

### 보고 지표
| 평가셋 | 지표 | 의미 |
|---|---|---|
| ToMBench test 30% | Accuracy + 31 abilities별 | In-domain 학습 효과 |
| OpenToM | **macro-F1** + Accuracy | OOD 일반화 (라벨 불균형) |
| ToMi | Accuracy | 합성 데이터 일반화 |
| SocialIQa | Accuracy | 다른 도메인 일반화 |
| HiToM | Accuracy | 고차 ToM 일반화 |

## 사용법
1. **런타임 → L4 GPU 설정** (Colab Pro 권장 · 24GB VRAM)
2. **런타임 → 모두 실행** (`Ctrl+F9`)
3. Drive 권한 팝업 → 허용

⏱ 약 **1.5~2.5시간** 소요 (L4 기준 · baseline 평가까지 포함)
   · T4로 실행 시 약 2.5~4시간

## ⚠️ 라이선스 / 윤리
비상업 학술 연구용. ToMBench README 권고에 대한 limitation은 마지막 셀에 명시.

## ⚙️ 설정

In [ ]:
# === 빠른 테스트 ===
QUICK_TEST = False     # True: ability별 10개씩만 (~20분)

# === 학습 설정 ===
NUM_EPOCHS = 4
TEST_RATIO = 0.30      # 전체 → test (=30%)
VAL_RATIO_OF_TRAIN = 0.20   # train_val 중 val (=20%) → 최종 train 56%, val 14%, test 30%
EARLY_STOPPING_PATIENCE = 2  # val_loss가 N epoch 동안 개선 안 되면 중단

# === Cross-eval 선택 ===
EVAL_OPENTOM   = True
EVAL_TOMI      = True
EVAL_SOCIALIQA = True
EVAL_HITOM     = True

# === 평가 샘플 캡 (시간 절약) ===
EVAL_MAX_PER_DATASET = 1000

# === Drive 저장 ===
DRIVE_FOLDER = 'ToMBench_연구결과'

# === 데이터 경로 ===
TOMBENCH_DATA_PATHS = [
    '/content/ToMBench/data', '/content/data',
    '/content/drive/MyDrive/ToMBench-main/data',
    '/content/drive/MyDrive/ToMBench/data',
]
OPENTOM_DATA_PATHS = [
    '/content/OpenToM/data',
    '/content/drive/MyDrive/OpenToM-main/data',
    '/content/drive/MyDrive/OpenToM/data',
]

## 🔧 [1/14] 환경 설정

In [ ]:
import sys, os, subprocess, time
print('=' * 60); print('🔧 [1/14] 환경 설정 (간소화 버전)'); print('=' * 60)

# === GPU 확인 ===
import torch
if not torch.cuda.is_available():
    print('\n❌ GPU 미설정. 런타임 → 런타임 유형 변경 → L4 GPU (Pro) 또는 T4 GPU'); raise SystemExit
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'✅ GPU: {gpu_name} ({vram_gb:.1f}GB VRAM)')
if 'L4' in gpu_name:
    print('   🚀 L4 감지 — batch_size=4, grad_accum=2 자동 적용')
elif 'A100' in gpu_name:
    print('   🚀 A100 감지 — batch_size=8 자동 적용')
elif 'T4' in gpu_name:
    print('   ⚠️ T4 감지 — batch_size=2 자동 적용')

# === 라이브러리 설치 (unsloth PyPI wheel — 빠르고 안정적) ===
print('\n📦 [1/2] unsloth + 학습 라이브러리 (PyPI wheel, 2~3분 예상)...')
t0 = time.time()
!pip install -q unsloth
print(f'   ⏱ {time.time()-t0:.0f}초')

print('\n📦 [2/2] datasets<4.0 + sklearn + pandas (1분 예상)...')
t0 = time.time()
!pip install -q "datasets<4.0.0" scikit-learn pandas tqdm
print(f'   ⏱ {time.time()-t0:.0f}초')

# === 버전 확인 ===
print('\n🔍 설치 버전 확인:')
import importlib
for pkg in ['unsloth','torch','transformers','trl','peft','accelerate','bitsandbytes','datasets']:
    try:
        m = importlib.import_module(pkg)
        print(f'   {pkg}: {getattr(m,"__version__","?")}')
    except Exception as e:
        print(f'   {pkg}: ❌ {type(e).__name__}')

print('\n✅ 라이브러리 설치 완료')

# === Drive 연결 ===
print('\n🔐 Drive 연결...')
print('👉 팝업 → 본인 계정 → 허용')
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = f'/content/drive/MyDrive/{DRIVE_FOLDER}'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'✅ Drive: {DRIVE_DIR}')
print('\n✨ [1/14] 완료\n')

## 📥 [2/14] ToMBench 로드 + **3분할** (Train / Val / Test)

In [ ]:
import json, glob, math, re, random
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

print('=' * 60); print('📥 [2/14] ToMBench 3-way split'); print('=' * 60)

TOMBENCH_DIR = None
for p in TOMBENCH_DATA_PATHS:
    if os.path.isdir(p) and glob.glob(os.path.join(p, '*.jsonl')):
        TOMBENCH_DIR = p; break
if TOMBENCH_DIR is None:
    print('GitHub에서 ToMBench 클론...')
    subprocess.run(['git','clone','--depth','1','https://github.com/zhchen18/ToMBench.git','/content/ToMBench'], check=False)
    TOMBENCH_DIR = '/content/ToMBench/data'
assert os.path.isdir(TOMBENCH_DIR), f'ToMBench 데이터 없음: {TOMBENCH_DATA_PATHS}'
print(f'✅ ToMBench: {TOMBENCH_DIR}')

def is_valid(x):
    if x is None: return False
    if isinstance(x, float) and math.isnan(x): return False
    s = str(x).strip()
    return len(s) > 0 and s.lower() != 'nan'

def normalize_ability(ab):
    return ab.replace('Non-literal', 'Non-Literal')

def build_user_prompt(story, question, opts):
    lines = [f'{l}. {t}' for l, t in opts]
    return (f'[Story]\n{story}\n\n[Question]\n{question}\n\n[Candidate Answers]\n' + '\n'.join(lines))

records = []
for fp in sorted(glob.glob(os.path.join(TOMBENCH_DIR, '*.jsonl'))):
    task = os.path.splitext(os.path.basename(fp))[0]
    with open(fp, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            row = json.loads(line)
            ability = normalize_ability(row.get('能力\nABILITY', 'Unknown'))
            category = ability.split(':')[0].strip()
            ans = row.get('答案\nANSWER') or row.get('ANSWER')
            if not is_valid(row.get('STORY')) or not is_valid(row.get('QUESTION')) or not is_valid(ans): continue
            opts = []
            for letter in ['A','B','C','D']:
                v = row.get(f'OPTION-{letter}')
                if is_valid(v): opts.append((letter, str(v).strip()))
            records.append({
                'source': 'ToMBench',
                'task': task, 'category': category, 'ability': ability,
                'user_prompt': build_user_prompt(row['STORY'], row['QUESTION'], opts),
                'answer': str(ans).strip().upper()[0],
            })

df = pd.DataFrame(records)
print(f'✅ ToMBench 총 {len(df)}개 / ability {df["ability"].nunique()}개')

if QUICK_TEST:
    df = df.groupby('ability', group_keys=False).apply(lambda g: g.sample(min(len(g), 10), random_state=42)).reset_index(drop=True)
    print(f'⚡ QUICK_TEST: {len(df)}개로 축소')

# ============================================================
# 🧪 종합 개선 Split: 검증 리포트 + 다중 라벨 표시 + 소량 우대
#                  + 정답 분포 균형 체크 + CSV 저장
# ============================================================

# --- (1) 분할 전 분포 분석 ---
print('\n📋 분할 전 분포 분석:')
cat_counts = df['category'].value_counts()
task_counts = df['task'].value_counts()
ab_counts = df['ability'].value_counts()
print(f'  카테고리 {len(cat_counts)}개: ' + ', '.join(f'{c}({n})' for c,n in cat_counts.items()))
print(f'  Task {len(task_counts)}개 / Ability {len(ab_counts)}개')

# 다중 ability 라벨 식별 (별도 분리 안 하고 표시만)
multi_label = [ab for ab in ab_counts.index if ab.count(':') > 1]
if multi_label:
    print(f'\n  ℹ️  복합 ability 라벨 {len(multi_label)}개 (별도 ability로 취급):')
    for m in multi_label:
        print(f'      - {m} ({ab_counts[m]}개)')

# --- (2) 소량 ability 식별 → 우대 분할 (60/20/20) ---
SMALL_THRESHOLD = 30
small_ab = ab_counts[ab_counts < SMALL_THRESHOLD].index.tolist()
large_ab = ab_counts[ab_counts >= SMALL_THRESHOLD].index.tolist()
if small_ab:
    print(f'\n  ⚠️  소량 ability {len(small_ab)}개 (<{SMALL_THRESHOLD}개) — 우대 분할(60/20/20) 적용:')
    for ab in small_ab:
        print(f'      - {ab} ({ab_counts[ab]}개)')

# --- (3) 큰 ability: ability stratified 56/14/30 분할 ---
large_df = df[df['ability'].isin(large_ab)].reset_index(drop=True)
if len(large_df) > 0:
    train_val_l, test_l = train_test_split(
        large_df, test_size=TEST_RATIO, random_state=42,
        stratify=large_df['ability'],
    )
    train_l, val_l = train_test_split(
        train_val_l, test_size=VAL_RATIO_OF_TRAIN, random_state=42,
        stratify=train_val_l['ability'],
    )
else:
    train_l = val_l = test_l = pd.DataFrame()

# --- (4) 작은 ability: 각각 60/20/20 (val/test 최소 보장) ---
parts = {'train': [], 'val': [], 'test': []}
for ab in small_ab:
    ab_df = df[df['ability'] == ab].reset_index(drop=True)
    n = len(ab_df)
    if n < 5:
        # 너무 적음: train에만
        parts['train'].append(ab_df)
        continue
    seed_ab = 42 + (hash(ab) % 100)
    tr, vt = train_test_split(ab_df, test_size=0.4, random_state=seed_ab)
    va, te = train_test_split(vt, test_size=0.5, random_state=seed_ab)
    parts['train'].append(tr); parts['val'].append(va); parts['test'].append(te)
train_s = pd.concat(parts['train']) if parts['train'] else pd.DataFrame()
val_s   = pd.concat(parts['val'])   if parts['val']   else pd.DataFrame()
test_s  = pd.concat(parts['test'])  if parts['test']  else pd.DataFrame()

train_df = pd.concat([train_l, train_s], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
val_df   = pd.concat([val_l, val_s],     ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
test_df  = pd.concat([test_l, test_s],   ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'\n✅ Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print(f'   비율: {len(train_df)/len(df):.1%} / {len(val_df)/len(df):.1%} / {len(test_df)/len(df):.1%}')

# --- (5) 검증 리포트: ability별 ---
ability_bd = pd.DataFrame({
    'Train': train_df['ability'].value_counts(),
    'Val':   val_df['ability'].value_counts(),
    'Test':  test_df['ability'].value_counts(),
}).fillna(0).astype(int)
ability_bd['Total'] = ability_bd.sum(axis=1)
ability_bd = ability_bd.sort_values('Total', ascending=False)
print(f'\n📋 Ability별 분할 (상위 10개 / 총 {len(ability_bd)}개):')
print(ability_bd.head(10).to_string())
if small_ab:
    small_bd = ability_bd.loc[ability_bd.index.intersection(small_ab)]
    print(f'\n📋 소량 ability 분할 결과 (Val 보장 확인):')
    print(small_bd.to_string())

# --- (6) 검증 리포트: Task별 (8개 균형) ---
task_bd = pd.DataFrame({
    'Train': train_df['task'].value_counts(),
    'Val':   val_df['task'].value_counts(),
    'Test':  test_df['task'].value_counts(),
}).fillna(0).astype(int)
task_bd['Total'] = task_bd.sum(axis=1)
task_bd = task_bd.sort_values('Total', ascending=False)
print(f'\n📋 Task별 분할 (총 {len(task_bd)}개):')
print(task_bd.to_string())

# --- (7) 검증 리포트: Category별 비율 ---
cat_bd = pd.DataFrame({
    'Train': train_df['category'].value_counts(),
    'Val':   val_df['category'].value_counts(),
    'Test':  test_df['category'].value_counts(),
}).fillna(0).astype(int)
cat_bd['Total'] = cat_bd.sum(axis=1)
for col in ['Train','Val','Test']:
    cat_bd[f'{col}%'] = (cat_bd[col] / cat_bd['Total'] * 100).round(1)
print(f'\n📋 Category별 분할 비율:')
print(cat_bd.to_string())

# --- (8) 정답 분포(A/B/C/D) 균형 체크 ---
print('\n📊 정답 분포 균형 체크:')
max_dev = 0
for split_name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    if len(split) == 0: continue
    ans_dist = split['answer'].value_counts(normalize=True).sort_index()
    pct_str = ' '.join(f'{l}={p:.1%}' for l, p in ans_dist.items())
    dev = ans_dist.max() - ans_dist.min() if len(ans_dist) > 1 else 0
    print(f'  {split_name}: {pct_str} (편차 {dev:.1%})')
    max_dev = max(max_dev, dev)
if max_dev < 0.10:
    print('  ✅ 균형 양호 (최대 편차 <10%)')
elif max_dev < 0.20:
    print('  ⚠️  약간 편향 (10~20%)')
else:
    print(f'  🚨 강한 편향 ({max_dev:.1%}) — 결과 해석 주의')

# --- (9) Drive 저장 (재현/검증용) ---
ability_bd.to_csv(f'{DRIVE_DIR}/split_ability_breakdown.csv')
task_bd.to_csv(f'{DRIVE_DIR}/split_task_breakdown.csv')
cat_bd.to_csv(f'{DRIVE_DIR}/split_category_breakdown.csv')
for nm, split in [('train',train_df),('val',val_df),('test',test_df)]:
    split[['source','task','category','ability','answer']].to_csv(
        f'{DRIVE_DIR}/split_{nm}_index.csv', index=True)
print(f'\n💾 split_*.csv 6개 저장됨 (재현/검증용)')

print('\n✨ [2/14] 완료\n')

## 🤖 [3/14] Qwen2.5-3B 4bit 로드

In [ ]:
print('=' * 60); print('🤖 [3/14] 모델 로드 (1~2분)'); print('=' * 60)

from unsloth import FastLanguageModel
MODEL_NAME = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH,
    dtype=None, load_in_4bit=True,
)
print('\n✨ [3/14] 완료 (LoRA는 baseline 평가 후 부착)\n')

## 🛠️ [4/14] 평가 헬퍼 + Cross-eval 데이터 로더

In [ ]:
from sklearn.metrics import f1_score

SYSTEM_PROMPT = (
    'Below is a multiple-choice question with a story and several answer options. '
    'Based on the content of the story and the given question, please infer the most likely answer '
    'and output the answer index in the format [[X]] where X is one of A, B, C, D, E.'
)

def messages_of(user_prompt, gold=None):
    m = [{'role':'system','content':SYSTEM_PROMPT}, {'role':'user','content':user_prompt}]
    if gold is not None: m.append({'role':'assistant','content':f'[[{gold}]]'})
    return m

def to_text(user_prompt, gold=None):
    return tokenizer.apply_chat_template(
        messages_of(user_prompt, gold), tokenize=False,
        add_generation_prompt=(gold is None))

ANS_RE = re.compile(r'\[\[\s*([A-E])\s*\]\]', re.IGNORECASE)
FB_RE = re.compile(r'\b([A-E])\b')

def extract_letter(text):
    m = ANS_RE.search(text)
    if m: return m.group(1).upper()
    m = FB_RE.search(text)
    if m: return m.group(1).upper()
    return 'A'

@torch.no_grad()
def evaluate(eval_records, model, tokenizer, desc='Eval'):
    from tqdm.auto import tqdm
    FastLanguageModel.for_inference(model)
    results = []
    for r in tqdm(eval_records, desc=desc):
        prompt = to_text(r['user_prompt'])
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        out = model.generate(**inputs, max_new_tokens=12, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
        gen = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        pred = extract_letter(gen)
        results.append({**r, 'pred': pred, 'raw': gen, 'correct': int(pred == r['answer'])})
    return pd.DataFrame(results)

def cap_eval(recs, n=EVAL_MAX_PER_DATASET):
    if n and len(recs) > n: return random.Random(42).sample(recs, n)
    return recs

# --- OpenToM ---
def load_opentom():
    OPENTOM_DIR = None
    for p in OPENTOM_DATA_PATHS:
        if os.path.isdir(p): OPENTOM_DIR = p; break
    if OPENTOM_DIR is None:
        subprocess.run(['git','clone','--depth','1','https://github.com/seacowx/OpenToM.git','/content/OpenToM'], check=False)
        if os.path.isdir('/content/OpenToM/data'): OPENTOM_DIR = '/content/OpenToM/data'
    if OPENTOM_DIR is None: print('⚠️ OpenToM 없음'); return []
    meta_path = os.path.join(OPENTOM_DIR, 'opentom_data', 'meta_data.json')
    recs = []
    if os.path.exists(meta_path):
        with open(meta_path) as f: meta = json.load(f)
        att_path = os.path.join(OPENTOM_DIR, 'opentom_data', 'attitude.json')
        if os.path.exists(att_path):
            with open(att_path) as f: att = json.load(f)
            opts_text = ['positive','negative','neutral']
            for sid, qs in att.items():
                narrative = meta.get(sid,{}).get('narrative','')
                if not narrative: continue
                for q in qs:
                    gold = q.get('answer','').strip().lower()
                    if gold not in opts_text: continue
                    shuffled = opts_text[:]
                    random.Random((hash(sid+q.get('question','')) & 0xffffffff)).shuffle(shuffled)
                    gold_letter = 'ABC'[shuffled.index(gold)]
                    recs.append({
                        'source':'OpenToM','task':'attitude','category':'Emotion/Attitude','ability':'Attitude',
                        'user_prompt':build_user_prompt(narrative, q['question'], list(zip('ABC',shuffled))),
                        'answer':gold_letter,
                    })
        cg_path = os.path.join(OPENTOM_DIR, 'opentom_data', 'location_cg_fo.json')
        if os.path.exists(cg_path):
            with open(cg_path) as f: cg = json.load(f)
            for sid, qs in cg.items():
                narrative = meta.get(sid,{}).get('narrative','')
                if not narrative: continue
                for q in qs:
                    gold = q.get('answer','').strip()
                    if gold not in ('Yes','No'): continue
                    opts = [('A','Yes'),('B','No')]
                    gold_letter = 'A' if gold == 'Yes' else 'B'
                    recs.append({
                        'source':'OpenToM','task':'location-cg','category':'Belief','ability':'Location False Beliefs',
                        'user_prompt': build_user_prompt(narrative, q['question'], opts),
                        'answer': gold_letter,
                    })
    return cap_eval(recs)

# --- ToMi ---
def load_tomi():
    TOMI_DIR = '/content/ToMi'
    if not os.path.isdir(TOMI_DIR):
        r = subprocess.run(['git','clone','--depth','1','https://github.com/facebookresearch/ToMi.git', TOMI_DIR], capture_output=True)
        if r.returncode != 0: print('⚠️ ToMi 클론 실패'); return []
    DATA_OUT = '/content/tomi_data'
    os.makedirs(DATA_OUT, exist_ok=True)
    if not os.path.exists(os.path.join(DATA_OUT,'test.txt')):
        r = subprocess.run([sys.executable,'main.py','-n','500','-o',DATA_OUT,'-s','42'], cwd=TOMI_DIR, capture_output=True)
        if r.returncode != 0: print('⚠️ ToMi 생성 실패'); return []
    samples = []; cur_facts, cur_qas = [], []; prev = 0
    with open(os.path.join(DATA_OUT,'test.txt')) as f:
        for line in f:
            line = line.rstrip('\n')
            if not line.strip(): continue
            m = re.match(r'^(\d+)\s(.*)$', line)
            if not m: continue
            num = int(m.group(1)); rest = m.group(2)
            if num <= prev and (cur_facts or cur_qas):
                narr = ' '.join(cur_facts)
                for q,a in cur_qas: samples.append({'narrative':narr,'question':q,'answer':a})
                cur_facts, cur_qas = [], []
            prev = num
            if '\t' in rest:
                parts = rest.split('\t')
                if len(parts) >= 2: cur_qas.append((parts[0].strip(), parts[1].strip()))
            else:
                cur_facts.append(rest.strip())
        if cur_facts or cur_qas:
            narr = ' '.join(cur_facts)
            for q,a in cur_qas: samples.append({'narrative':narr,'question':q,'answer':a})
    ans_pool = list({s['answer'] for s in samples})
    if len(ans_pool) < 4: return []
    recs = []
    for s in samples:
        gold = s['answer']
        distractors = [a for a in ans_pool if a != gold]
        ds = random.Random(hash(s['narrative'][:40]) & 0xffffffff).sample(distractors, 3)
        opts = [gold] + ds
        random.Random(hash(s['question']) & 0xffffffff).shuffle(opts)
        gold_letter = 'ABCD'[opts.index(gold)]
        recs.append({
            'source':'ToMi','task':'location-belief','category':'Belief','ability':'Location False Beliefs',
            'user_prompt': build_user_prompt(s['narrative'], s['question'], list(zip('ABCD', opts))),
            'answer': gold_letter,
        })
    return cap_eval(recs)

# --- SocialIQa ---
def load_socialiqa():
    from datasets import load_dataset
    ds = None
    # 1) HF 직접 시도 (datasets<4.0이면 작동)
    for name in ['allenai/social_i_qa','social_i_qa']:
        try:
            ds = load_dataset(name, split='validation'); print(f'✅ SocialIQa from HF: {name}'); break
        except Exception as e:
            try:
                ds = load_dataset(name, split='validation', trust_remote_code=True); print(f'✅ SocialIQa from HF (legacy): {name}'); break
            except Exception as e2:
                print(f'  ↳ {name}: {type(e2).__name__}')
                continue
    # 2) 직접 URL fallback (AllenAI 공식 zip)
    if ds is None:
        try:
            import urllib.request, zipfile
            url = 'https://storage.googleapis.com/ai2-mosaic/public/socialiqa/socialiqa-train-dev.zip'
            zip_path = '/content/socialiqa.zip'
            urllib.request.urlretrieve(url, zip_path)
            with zipfile.ZipFile(zip_path) as z: z.extractall('/content/socialiqa_data')
            import glob
            dev_jsonl = glob.glob('/content/socialiqa_data/**/dev.jsonl', recursive=True)
            dev_lbl   = glob.glob('/content/socialiqa_data/**/dev-labels.lst', recursive=True)
            if dev_jsonl and dev_lbl:
                with open(dev_jsonl[0]) as f: rows = [json.loads(l) for l in f]
                with open(dev_lbl[0]) as f: lbls = [l.strip() for l in f]
                ds = [dict(r, label=lbls[i]) for i,r in enumerate(rows)]
                print(f'✅ SocialIQa from AllenAI zip: {len(ds)}개')
        except Exception as e:
            print(f'  ↳ AllenAI zip 실패: {e}')
    if ds is None: print('⚠️ SocialIQa 로드 완전 실패 (스킵)'); return []
    recs = []
    for x in ds:
        ctx = x.get('context',''); q = x.get('question','')
        a,b,c = x.get('answerA',''), x.get('answerB',''), x.get('answerC','')
        lbl = str(x.get('label','')).strip()
        if lbl not in ('1','2','3'): continue
        gold_letter = {'1':'A','2':'B','3':'C'}[lbl]
        recs.append({
            'source':'SocialIQa','task':'social_cs','category':'Mixed','ability':'Mixed',
            'user_prompt': build_user_prompt(ctx, q, [('A',a),('B',b),('C',c)]),
            'answer': gold_letter,
        })
    return cap_eval(recs)

# --- HiToM ---
# 실제 schema (Hi-ToM/Hi-ToM_Dataset on HF):
#   story: str, question: str, answer: str (예: "green_drawer"),
#   choices: str (예: "A. blue_drawer, B. green_crate, C. red_bucket, ...")
# 정답은 문자열 → choices에서 매칭하여 letter 추출 필요
import re as _re_hitom

def _parse_hitom_choices(choices_str):
    if not isinstance(choices_str, str): return []
    pat = _re_hitom.compile(r'([A-Z])\.\s*([^,]+?)(?=,\s*[A-Z]\.|$)')
    return [(m.group(1).upper(), m.group(2).strip()) for m in pat.finditer(choices_str)]

def load_hitom():
    from datasets import load_dataset
    ds = None
    for name in ['Hi-ToM/Hi-ToM_Dataset', 'umwyf/Hi-ToM_Dataset']:
        try:
            ds = load_dataset(name, split='train'); print(f'✅ HiToM from {name}'); break
        except Exception as e:
            print(f'  ↳ {name}: {type(e).__name__}')
    if ds is None:
        try:
            HITOM_DIR = '/content/Hi-ToM_dataset'
            if not os.path.isdir(HITOM_DIR):
                subprocess.run(['git','clone','--depth','1','https://github.com/ying-hui-he/Hi-ToM_dataset.git', HITOM_DIR], capture_output=True, check=False)
            import glob
            json_files = glob.glob(f'{HITOM_DIR}/**/*.json', recursive=True) + glob.glob(f'{HITOM_DIR}/**/*.jsonl', recursive=True)
            if json_files:
                rows = []
                for fp in json_files:
                    with open(fp) as f:
                        try: data = json.load(f); rows.extend(data if isinstance(data, list) else [data])
                        except: f.seek(0); rows.extend(json.loads(l) for l in f if l.strip())
                if rows: ds = rows; print(f'✅ HiToM from GitHub clone: {len(rows)}개')
        except Exception as e:
            print(f'  ↳ GitHub clone 실패: {e}')
    if ds is None: print('⚠️ HiToM 자동 로드 실패 (스킵)'); return []

    recs = []
    skipped = 0
    LETTERS = 'ABCDEFGHIJKLMNO'
    for x in ds:
        story = x.get('story') or x.get('context') or x.get('narrative') or ''
        q = x.get('question') or ''
        choices_raw = x.get('choices') or x.get('options') or []
        gold_raw = x.get('answer')
        if gold_raw is None: gold_raw = x.get('label')
        if not story or not q or not choices_raw or gold_raw is None: skipped += 1; continue

        if isinstance(choices_raw, str):
            opts = _parse_hitom_choices(choices_raw)
            if not opts:
                try:
                    parsed = json.loads(choices_raw)
                    opts = list(zip(LETTERS, parsed))
                except:
                    parts = [p.strip() for p in choices_raw.split('|')]
                    opts = list(zip(LETTERS, parts))
        elif isinstance(choices_raw, list):
            opts = list(zip(LETTERS, choices_raw))
        else:
            skipped += 1; continue

        if not opts: skipped += 1; continue

        gold_str = str(gold_raw).strip()
        gold_letter = None
        if len(gold_str) == 1 and gold_str.upper() in LETTERS:
            gold_letter = gold_str.upper()
        else:
            for letter, text in opts:
                if text.lower() == gold_str.lower():
                    gold_letter = letter; break
            if gold_letter is None:
                for letter, text in opts:
                    if gold_str.lower() in text.lower() or text.lower() in gold_str.lower():
                        gold_letter = letter; break
        if gold_letter is None: skipped += 1; continue

        if len(opts) > 5:
            gold_opt = next(o for o in opts if o[0] == gold_letter)
            others = [o for o in opts if o[0] != gold_letter]
            sampled = random.Random(hash(story+q) & 0xffffffff).sample(others, min(4, len(others)))
            new_pool = [gold_opt] + sampled
            random.Random(hash(q) & 0xffffffff).shuffle(new_pool)
            opts = [('ABCDE'[i], t) for i,(_, t) in enumerate(new_pool)]
            for i,(_, t) in enumerate(new_pool):
                if t == gold_opt[1]: gold_letter = 'ABCDE'[i]; break

        order = x.get('question_order', None)
        ability_label = f'High-Order False Beliefs (order={order})' if order is not None else 'High-Order False Beliefs'
        recs.append({
            'source':'HiToM','task':'high-order','category':'Belief','ability':ability_label,
            'user_prompt': build_user_prompt(story, q, opts),
            'answer': gold_letter,
        })
    if skipped: print(f'  ↳ HiToM: {skipped}개 스킵 (답/선택지 매칭 실패)')
    return cap_eval(recs)

print('✅ 평가 헬퍼 + 로더 준비 완료')
print('\n✨ [4/14] 완료\n')

## 📦 [5/14] Cross-eval 데이터 미리 로드 (baseline·finetuned에서 공통 사용)

In [ ]:
print('=' * 60); print('📦 [5/14] 외부 평가 데이터 준비'); print('=' * 60)

eval_sets = {'ToMBench': test_df.to_dict('records')}
print(f'✅ ToMBench test: {len(eval_sets["ToMBench"])}개')

if EVAL_OPENTOM:
    eval_sets['OpenToM'] = load_opentom()
    print(f'✅ OpenToM: {len(eval_sets["OpenToM"])}개')
if EVAL_TOMI:
    eval_sets['ToMi'] = load_tomi()
    print(f'✅ ToMi: {len(eval_sets["ToMi"])}개')
if EVAL_SOCIALIQA:
    eval_sets['SocialIQa'] = load_socialiqa()
    print(f'✅ SocialIQa: {len(eval_sets["SocialIQa"])}개')
if EVAL_HITOM:
    eval_sets['HiToM'] = load_hitom()
    print(f'✅ HiToM: {len(eval_sets["HiToM"])}개')

# 비어있는 셋 제거
eval_sets = {k:v for k,v in eval_sets.items() if len(v) > 0}
print(f'\n📌 평가 예정 데이터셋: {list(eval_sets.keys())}')
print('\n✨ [5/14] 완료\n')

## 📊 [6/14] **Baseline 측정** (학습 전 zero-shot)

모든 평가 데이터셋에 대해 학습 전 Qwen2.5-3B 점수 측정.

In [ ]:
print('=' * 60); print('📊 [6/14] Baseline 평가 (학습 전)'); print('=' * 60)

baseline = {}
baseline_dfs = {}

for name, recs in eval_sets.items():
    print(f'\n--- {name} ---')
    df_r = evaluate(recs, model, tokenizer, desc=f'Baseline {name}')
    acc = df_r['correct'].mean()
    metrics = {'accuracy': acc}
    if name == 'OpenToM':
        # macro-F1 추가
        try:
            f1 = f1_score(df_r['answer'], df_r['pred'], average='macro', zero_division=0)
            metrics['macro_f1'] = f1
        except Exception as e:
            print(' F1 계산 실패:', e)
    baseline[name] = metrics
    baseline_dfs[name] = df_r
    df_r.to_csv(f'{DRIVE_DIR}/baseline_{name}.csv', index=False)
    print(f'  Accuracy: {acc:.4f}' + (f' | macro-F1: {metrics["macro_f1"]:.4f}' if 'macro_f1' in metrics else ''))

print('\n📋 Baseline 요약:')
for k,v in baseline.items(): print(f'  {k}: {v}')
print('\n✨ [6/14] 완료\n')

## 🧩 [7/14] LoRA 어댑터 부착

In [ ]:
print('=' * 60); print('🧩 [7/14] LoRA 어댑터 부착'); print('=' * 60)

model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16, lora_dropout=0.0, bias='none',
    use_gradient_checkpointing='unsloth', random_state=42,
)
print('✅ LoRA 부착 완료')
print('\n✨ [7/14] 완료\n')

## 🔥 [8/14] 학습 (Val로 early stopping)

- 매 epoch마다 val_loss 측정
- `EARLY_STOPPING_PATIENCE` epoch 동안 개선 없으면 중단
- 최종적으로 best val_loss 시점 모델 로드

In [ ]:
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback
from unsloth import is_bfloat16_supported

print('=' * 60); print(f'🔥 [8/14] 학습 시작 (max {NUM_EPOCHS} epochs, EarlyStop patience={EARLY_STOPPING_PATIENCE})'); print('=' * 60)

# === GPU별 batch size 자동 설정 (effective batch=8 유지) ===
_gpu = torch.cuda.get_device_name(0)
_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
if 'A100' in _gpu or _vram >= 35:
    TRAIN_BS, GRAD_ACCUM, EVAL_BS = 8, 1, 16   # A100 40GB+
elif 'L4' in _gpu or _vram >= 22:
    TRAIN_BS, GRAD_ACCUM, EVAL_BS = 4, 2, 8    # L4 24GB
else:
    TRAIN_BS, GRAD_ACCUM, EVAL_BS = 2, 4, 4    # T4 16GB
print(f'📐 GPU={_gpu} ({_vram:.0f}GB) → train_bs={TRAIN_BS} × grad_accum={GRAD_ACCUM} (effective={TRAIN_BS*GRAD_ACCUM}), eval_bs={EVAL_BS}')

train_texts = [to_text(r['user_prompt'], r['answer']) for r in train_df.to_dict('records')]
val_texts   = [to_text(r['user_prompt'], r['answer']) for r in val_df.to_dict('records')]
train_ds_text = Dataset.from_dict({'text': train_texts})
val_ds_text   = Dataset.from_dict({'text': val_texts})
print(f'Train: {len(train_ds_text)} | Val: {len(val_ds_text)}')

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=train_ds_text,
    eval_dataset=val_ds_text,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2, packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=TRAIN_BS,
        per_device_eval_batch_size=EVAL_BS,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_ratio=0.03,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=20,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        save_total_limit=2,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=42,
        output_dir='outputs',
        report_to='none',
    ),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

t0 = time.time()
stats = trainer.train()
print(f'\n✅ 학습 완료 ({(time.time()-t0)/60:.1f}분) | final train loss: {stats.training_loss:.4f}')

# 학습 history 저장
log_df = pd.DataFrame(trainer.state.log_history)
log_df.to_csv(f'{DRIVE_DIR}/training_log.csv', index=False)
print('📈 training_log.csv 저장됨')

# 어댑터 Drive 저장
import shutil
local_save = '/content/qwen25_3b_tombench_lora'
model.save_pretrained(local_save); tokenizer.save_pretrained(local_save)
drive_save = f'{DRIVE_DIR}/lora_adapter'
if os.path.exists(drive_save): shutil.rmtree(drive_save)
shutil.copytree(local_save, drive_save)
print(f'💾 Drive: {drive_save}')
print('\n✨ [8/14] 완료\n')

## 🎯 [9/14] **Finetuned 평가** — 모든 평가셋

In [ ]:
print('=' * 60); print('🎯 [9/14] Finetuned 평가 (학습 후)'); print('=' * 60)

finetuned = {}
finetuned_dfs = {}

for name, recs in eval_sets.items():
    print(f'\n--- {name} ---')
    df_r = evaluate(recs, model, tokenizer, desc=f'Finetuned {name}')
    acc = df_r['correct'].mean()
    metrics = {'accuracy': acc}
    if name == 'OpenToM':
        try:
            f1 = f1_score(df_r['answer'], df_r['pred'], average='macro', zero_division=0)
            metrics['macro_f1'] = f1
        except Exception as e:
            print(' F1 계산 실패:', e)
    finetuned[name] = metrics
    finetuned_dfs[name] = df_r
    df_r.to_csv(f'{DRIVE_DIR}/finetuned_{name}.csv', index=False)
    print(f'  Accuracy: {acc:.4f}' + (f' | macro-F1: {metrics["macro_f1"]:.4f}' if 'macro_f1' in metrics else ''))

print('\n✨ [9/14] 완료\n')

## 🔬 [10/14] ToMBench 31 abilities 세분화 분석 (in-domain)

In [ ]:
print('=' * 60); print('🔬 [10/14] ToMBench ability별 분석'); print('=' * 60)

df_b = baseline_dfs['ToMBench'].copy()
df_f = finetuned_dfs['ToMBench'].copy()
df_b['model'] = 'baseline'; df_f['model'] = 'finetuned'

by_ab_b = df_b.groupby('ability')['correct'].agg(['mean','count']).rename(columns={'mean':'baseline_acc'})
by_ab_f = df_f.groupby('ability')['correct'].agg(['mean']).rename(columns={'mean':'finetuned_acc'})
by_ab = by_ab_b.join(by_ab_f)
by_ab['delta'] = by_ab['finetuned_acc'] - by_ab['baseline_acc']
by_ab = by_ab.sort_values('delta', ascending=False)
by_ab.to_csv(f'{DRIVE_DIR}/ability_breakdown.csv')

print('\n📊 가장 많이 향상된 ability TOP 10:')
print(by_ab.head(10).to_string())
print('\n📊 가장 적게 향상/하락한 ability:')
print(by_ab.tail(10).to_string())

# Category 별
by_cat_b = df_b.groupby('category')['correct'].mean().rename('baseline_acc')
by_cat_f = df_f.groupby('category')['correct'].mean().rename('finetuned_acc')
by_cat = pd.concat([by_cat_b, by_cat_f], axis=1)
by_cat['delta'] = by_cat['finetuned_acc'] - by_cat['baseline_acc']
by_cat = by_cat.sort_values('delta', ascending=False)
by_cat.to_csv(f'{DRIVE_DIR}/category_breakdown.csv')
print('\n📊 카테고리별 Δ:')
print(by_cat.to_string())
print('\n✨ [10/14] 완료\n')

## 🏆 [11/14] 최종 비교 표 (Δ 측정)

In [ ]:
print('=' * 60); print('🏆 [11/14] 최종 비교 표'); print('=' * 60)

rows = []
for name in eval_sets.keys():
    b = baseline.get(name, {})
    f = finetuned.get(name, {})
    rows.append({
        'Dataset': name,
        'Baseline_Acc': b.get('accuracy', None),
        'Finetuned_Acc': f.get('accuracy', None),
        'Δ_Acc': (f.get('accuracy',0) - b.get('accuracy',0)) if b and f else None,
        'Baseline_F1': b.get('macro_f1', None),
        'Finetuned_F1': f.get('macro_f1', None),
        'Δ_F1': (f.get('macro_f1',0) - b.get('macro_f1',0)) if b.get('macro_f1') is not None and f.get('macro_f1') is not None else None,
        'N': len(eval_sets[name]),
    })
summary_df = pd.DataFrame(rows)
summary_df.to_csv(f'{DRIVE_DIR}/SUMMARY_main_table.csv', index=False)

print('\n' + '='*70)
print('🎉 최종 결과 표 (논문 Table 1 후보)')
print('='*70)
with pd.option_context('display.float_format','{:,.4f}'.format):
    print(summary_df.to_string(index=False))

print('\n📌 해석 가이드:')
print('  - ToMBench Δ > 0.1: In-domain 학습 효과 큼')
print('  - OpenToM/ToMi/HiToM Δ > 0: ToM 일반화 성공')
print('  - SocialIQa Δ ≈ 0 or 음수: ToMBench 학습이 social commonsense에 transfer 안 됨 (보고할 만한 결과)')

print('\n✨ [11/14] 완료\n')

## 📈 [12/14] 시각화 (선택) — 결과 그래프

In [ ]:
import matplotlib.pyplot as plt

print('=' * 60); print('📈 [12/14] 시각화'); print('=' * 60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1) 데이터셋별 baseline vs finetuned
x = np.arange(len(summary_df))
w = 0.35
axes[0].bar(x - w/2, summary_df['Baseline_Acc'], w, label='Baseline (zero-shot)', alpha=0.7)
axes[0].bar(x + w/2, summary_df['Finetuned_Acc'], w, label='Finetuned (ToMBench)', alpha=0.9)
axes[0].set_xticks(x); axes[0].set_xticklabels(summary_df['Dataset'], rotation=15)
axes[0].set_ylabel('Accuracy'); axes[0].set_title('Baseline vs Finetuned by Dataset')
axes[0].legend(); axes[0].grid(alpha=0.3)

# 2) ToMBench category별 Δ
by_cat_sorted = by_cat.sort_values('delta')
colors = ['red' if d < 0 else 'green' for d in by_cat_sorted['delta']]
axes[1].barh(by_cat_sorted.index, by_cat_sorted['delta'], color=colors, alpha=0.7)
axes[1].axvline(0, color='black', linewidth=0.5)
axes[1].set_xlabel('Δ Accuracy (finetuned - baseline)')
axes[1].set_title('ToMBench Category Improvement')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{DRIVE_DIR}/figure_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 figure_summary.png 저장됨')
print('\n✨ [12/14] 완료\n')

## 💾 [13/14] 모든 결과 파일 목록

In [ ]:
print('=' * 60); print(f'💾 [13/14] Drive 저장 파일'); print('=' * 60)
print(f'📁 {DRIVE_DIR}/')
for f in sorted(os.listdir(DRIVE_DIR)):
    path = os.path.join(DRIVE_DIR, f)
    if os.path.isdir(path):
        print(f'   📁 {f}/')
    else:
        sz = os.path.getsize(path) / 1024
        print(f'   📄 {f} ({sz:.1f} KB)')
print(f'\n🌐 https://drive.google.com → MyDrive/{DRIVE_FOLDER}')
print('\n✨ [13/14] 완료\n')

## 📝 [14/14] 논문 보고용 메모

In [ ]:
memo = f"""
================================================================
📝 논문 방법론 / 결과 요약 (이 셀 출력을 그대로 복사 가능)
================================================================

## Method
- Base model: {MODEL_NAME}
- Fine-tuning: QLoRA (r=16, alpha=16, dropout=0.0)
- Library: Unsloth + TRL SFTTrainer
- Data split (ToMBench, N={len(df)}):
  * Train: {len(train_df)} ({len(train_df)/len(df):.1%})
  * Val:   {len(val_df)} ({len(val_df)/len(df):.1%}) — early stopping
  * Test:  {len(test_df)} ({len(test_df)/len(df):.1%}) — sealed, evaluated only at end
  * Stratified by ATOMS ability (seed=42)
- Training: max {NUM_EPOCHS} epochs, lr=2e-4, cosine, AdamW-8bit
- Early stopping: patience={EARLY_STOPPING_PATIENCE} on eval_loss

## Evaluation Datasets
- In-domain: ToMBench test split (N={len(test_df)})
- Cross-benchmark:
"""
for name in eval_sets:
    if name != 'ToMBench':
        memo += f'  * {name} (N={len(eval_sets[name])})\n'
memo += f"""
## Main Results Table (also saved as SUMMARY_main_table.csv)
"""
memo += summary_df.to_string(index=False) + '\n'

memo += '''
## Limitation Statement (논문 Limitations 섹션에 명시 권장)

We acknowledge that the ToMBench authors recommend against using their benchmark
for training to prevent data contamination. Our use is a controlled internal
experiment with strict 70/30 train/test separation (stratified by ATOMS ability,
seed=42), and we do not compare our ToMBench scores against models evaluated on
the full benchmark. Our ToMBench numbers reflect only our held-out 30% split.
The primary contribution is the cross-benchmark transfer evaluation on OpenToM,
ToMi, SocialIQa, and HiToM, which measures whether high-quality, high-reasoning
ToM training data generalizes across diverse ToM evaluation paradigms.

## Citation (BibTeX skeleton)

@misc{chen2024tombench, ... }            % ToMBench
@inproceedings{xu2024opentom, ... }       % OpenToM
@inproceedings{le2019revisiting, ... }    % ToMi
@inproceedings{sap2019socialiqa, ... }    % SocialIQa
@inproceedings{wu2023hitom, ... }         % HiToM
@misc{qwen2.5, ... }                       % Qwen2.5
% Unsloth: https://github.com/unslothai/unsloth
'''

print(memo)

# Drive에도 저장
with open(f'{DRIVE_DIR}/paper_methodology_notes.txt', 'w') as f:
    f.write(memo)
print('💾 paper_methodology_notes.txt 저장됨')
print('\n🎉 전체 파이프라인 완료!\n')

---

## 🔁 학습한 모델 재사용

```python
from google.colab import drive
drive.mount('/content/drive')
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    '/content/drive/MyDrive/ToMBench_연구결과/lora_adapter',
    max_seq_length=2048, load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
```

## ❓ 오류 발생 시
- `[N/14]` 단계 번호 + 메시지 확인
- OOM → `per_device_train_batch_size=1`로
- 데이터셋 로드 실패 → `datasets<4.0.0` pin이 적용됐는지 확인 (런타임 재시작 필요)
- HiToM/SocialIQa 둘 다 실패 → 평가셋에서 자동 제외되며, ToMBench·OpenToM·ToMi 만으로도 논문 결과 산출 가능

## 📌 데이터셋 출처 (재현용)
- **ToMBench**: `github.com/zhchen18/ToMBench` (ACL 2024) — 로컬 jsonl 사용
- **OpenToM**: `github.com/seacowx/OpenToM` (CC-BY-NC 4.0, train 금지 명시) — 평가 only ✅
- **ToMi**: `github.com/facebookresearch/ToMi` (절차적 생성, CC-BY-NC 4.0) — 평가 only ✅
- **SocialIQa**: `allenai/social_i_qa` (CC-BY 4.0, 학습 자유) — 평가만
- **HiToM**: `Hi-ToM/Hi-ToM_Dataset` HF (15지선다 → 5지로 축소 후 평가)
